
## **Cargar WebscrapingPandas 🐍**
Pandas es una biblioteca de Python de código abierto que se utiliza para la manipulación, el análisis y la limpieza de datos. Proporciona herramientas rápidas y flexibles para trabajar con datos tabulares, de forma similar a las hojas de cálculo o las tablas SQL.

Pandas se utiliza en ciencia de datos y análisis debido a su integración con bibliotecas como:

* NumPy : operaciones numéricas
* Matplotlib y Seaborn : visualización de datos
* SciPy : análisis estadístico
* Scikit-learn : flujos de trabajo de aprendizaje automático


In [331]:
import pandas as pd
import requests
from io import StringIO

url = "https://es.wikipedia.org/wiki/Anexo:Pa%C3%ADses_y_territorios_dependientes_por_poblaci%C3%B3n"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
print("codigo", response.status_code)

response.raise_for_status()
table = pd.read_html(StringIO(response.text))

df_web = table[0]
print(df_web.head())

codigo 200
  N.º País (o territorio dependiente)  \
0   1                           India   
1   2                       China[12]   
2   3                  Estados Unidos   
3   4                       Indonesia   
4   5                        Pakistán   

  Proyección exponencial de la población al 1/7/2026[7] Total mun- dial (%)  \
0                                      1 429 404 000                   1751   
1                                      1 403 203 000                   1719   
2                                        343 467 000                    421   
3                                        290 069 000                    355   
4                                        262 766 000                    322   

  Cambio medio anual (%)[8] Cambio absoluto anual promedio  \
0                       084                     12 020 000   
1                      -024                     -3 390 000   
2                       049                      1 692 000   
3                  

### 2. Revisar valores faltantes

In [332]:
# Identificar valores nulos
df_web.isnull().sum()

df_web.info()
df_web.describe()
print(df_web.columns)

<class 'pandas.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 12 columns):
 #   Column                                                                                             Non-Null Count  Dtype
---  ------                                                                                             --------------  -----
 0   N.º                                                                                                245 non-null    str  
 1   País (o territorio dependiente)                                                                    246 non-null    str  
 2   Proyección exponencial de la población al 1/7/2026[7]                                              246 non-null    str  
 3   Total mun- dial (%)                                                                                246 non-null    str  
 4   Cambio medio anual (%)[8]                                                                          246 non-null    str  
 5   Cambio absoluto anu

### 3. Codificar una variable categórica

In [333]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df_web['tipo_label'] = le.fit_transform(df_web['Tipo[11]'])

df_web[['Tipo[11]', 'tipo_label']].head()



from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

tipo_encoded = encoder.fit_transform(df_web[['Tipo[11]']])

pd.DataFrame(
    tipo_encoded,
    columns=encoder.get_feature_names_out(['Tipo[11]'])
).head()


,Tipo[11]_A,Tipo[11]_B,Tipo[11]_C,Tipo[11]_D,Tipo[11]_E,Tipo[11]_F,Tipo[11]_M,Tipo[11]_O,Tipo[11]_P,Tipo[11]_R,Tipo[11]_T,Tipo[11]_Tipo
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 4. Normalización y estandarización

In [334]:
# Convertir la población a número

df_web['poblacion'] = (
    df_web['Proyección exponencial de la población al 1/7/2026[7]']
    .astype(str)
    .str.replace('\xa0', '', regex=False)
)

df_web['poblacion'] = pd.to_numeric(
    df_web['poblacion'],
    errors='coerce'
)

# Normalización y estandarización

from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Normalización
normalizador = MinMaxScaler()

df_web['poblacion_normalizada'] = normalizador.fit_transform(
    df_web[['poblacion']]
)

# Estandarización
estandarizador = StandardScaler()

df_web['poblacion_estandarizada'] = estandarizador.fit_transform(
    df_web[['poblacion']]
)

# Mostrar resultados
df_web[
    ['poblacion',
     'poblacion_normalizada',
     'poblacion_estandarizada']
].head()

,poblacion,poblacion_normalizada,poblacion_estandarizada
0,1.429404e+09,0.175117,2.546068
1,1.403203e+09,0.171907,2.497116
2,3.434670e+08,0.042078,0.517209
3,2.900690e+08,0.035536,0.417445
4,2.627660e+08,0.032192,0.366435


### 4.SMOTE

In [363]:
print(y_balanceado.value_counts())

tipo_label
0     104
4     104
9     104
2     104
6     104
7     104
10    104
5     104
8     104
3     104
Name: count, dtype: int64
